# ORB export audit — 14 September 2026

## tl;dr

44 workbooks represent 41 distinct runs, not 44 independent confirmations. The old results suggest research candidates, not a proven winner. Paper trading has 10 closed trades, 5 winners, and +ZAR 4,890.11 reconciled to the balance ledger. The new Pine filters and 90-minute delay must be tested prospectively.

This notebook reproduces evidence for `README.md`; it does not execute Pine or run a new price-series backtest.

## Context & Methods

### Key Assumptions

- Export display timestamps are Africa/Johannesburg (UTC+2), confirmed by the user; session calculations use each workbook's input timezone.
- A trade's PnL is duplicated on entry and exit rows: group by trade number and count the closed exit once. Exclude `Open` exit placeholders from closed performance.
- Budget-normalized result = net PnL / (reconstructed pre-entry equity × risk percentage). This is not exact trade R or a rerun at lower risk.
- Comparisons use within-pair overlapping history. Carried-in trades are excluded. The retrospective 70/30 split is not an untouched out-of-sample test.
- PF is undefined when there is no gross loss, not zero or a guaranteed infinite edge. Binomial intervals do not account for correlated trades or parameter-selection bias.

Dependencies: Python 3, openpyxl, pandas, nbformat and a Jupyter Python kernel. Source files remain read-only. Outputs are regenerated from the analysis script in `tools/`.

## Data

### 1. Locate inputs and regenerate evidence

Run from this notebook directory or the Trading Board workspace. The subprocess uses the same Python interpreter as the notebook kernel.

In [1]:
from pathlib import Path
import sys, subprocess, json
import pandas as pd
root = Path.cwd()
if not (root / "tools/analyze_orb_results.py").exists():
    root = root.parent
assert (root / "Strategy Tester Results").is_dir(), "Run from the Trading Board workspace or research folder"
result = subprocess.run([sys.executable, str(root / "tools/analyze_orb_results.py")], capture_output=True, text=True, check=True)
evidence = json.loads((root / "ORB Research 2026-09-14/analysis_evidence.json").read_text())
print(f"Read {len(evidence['runs'])} workbooks; {evidence['unique_runs']} distinct runs. Export timezone: {evidence['export_timezone']}.")

Read 44 workbooks; 41 distinct runs. Export timezone: Africa/Johannesburg, confirmed by user.


### 2. Reconcile grain, source hashes and paper balance

The supplementary checks include source parity, risk formulas, range coverage examples and export reconciliation. They are not a local Pine interpreter.

In [2]:
check = subprocess.run([sys.executable, str(root / "tools/test_orb_research.py")], capture_output=True, text=True, check=True)
print(check.stderr[-300:])
assert all(not run["issues"] for run in evidence["runs"])
paper = evidence["paper"]
assert abs(paper["reconciliation"]) < 0.001
pd.DataFrame([{"workbooks":len(evidence["runs"]), "unique_runs":evidence["unique_runs"], "open_positions_excluded":sum(len(r["open_trades"]) for r in evidence["runs"]), "paper_closed_trades":paper["closed_trades"]["n"], "paper_balance_ZAR":paper["balance_last"], "paper_net_ZAR":paper["closed_trades"]["net"]}])

s_unchanged (__main__.SourceTests.test_originals_unchanged) ... ok
test_shared_engine (__main__.SourceTests.test_shared_engine) ... ok
test_tester_model (__main__.SourceTests.test_tester_model) ... ok

----------------------------------------------------------------------
Ran 19 tests in 0.017s

OK



,workbooks,unique_runs,open_positions_excluded,paper_closed_trades,paper_balance_ZAR,paper_net_ZAR
0,44,41,5,10,14890.109623,4890.11


## Results

### 3. Review the candidate runs

These are old v1 results at 10% risk and 2.75R with placeholder costs. Different history lengths and session exits prevent an unqualified full-period ranking. Candidate rows are chosen for follow-up, not independently validated.

In [3]:
comparison = pd.read_csv(root / "ORB Research 2026-09-14/run_comparison.csv")
candidate = ((comparison.pair.eq("XAUUSD") & comparison.direction.eq("Short only") & comparison.timeframe.eq("15 minutes")) | (comparison.pair.eq("EURUSD") & comparison.direction.eq("Long only") & comparison.timeframe.eq("5 minutes")) | (comparison.pair.eq("GBPJPY") & comparison.direction.eq("Short only") & comparison.timeframe.eq("5 minutes")) | (comparison.pair.eq("GBPUSD") & comparison.direction.eq("Both") & comparison.timeframe.eq("5 minutes") & comparison.model.eq("Break + retest")))
duplicate_files = {r["file"] for r in evidence["runs"] if r["duplicate_of"]}
comparison.loc[candidate & ~comparison.file.isin(duplicate_files), ["pair","symbol","timeframe","direction","model","trades","win_pct","pf","dd_pct","common_trades","common_pf"]].round(3)

,pair,symbol,timeframe,direction,model,trades,win_pct,pf,dd_pct,common_trades,common_pf
8,EURUSD,DERIV:EURUSD,5 minutes,Long only,Close breakout,21,57.143,1.816,28.56,21,2.128
18,GBPJPY,FX:GBPJPY,5 minutes,Short only,Close breakout,17,64.706,2.344,30.10,17,2.923
21,GBPUSD,FX:GBPUSD,5 minutes,Both,Break + retest,20,45.000,1.466,39.88,20,1.671
33,XAUUSD,FX:XAUUSD,15 minutes,Short only,Break + retest,33,45.455,1.444,45.53,11,1.268


### 4. Compare early and later entries without claiming causality

Filtering existing trades cannot simulate setups the old strategy did not take. Here PF is computed from budget-normalized PnL.

In [4]:
timing=[]
for run in evidence["runs"]:
    if run["duplicate_of"]: continue
    selected = (run["pair"] == "XAUUSD" and run["props"]["Timeframe"] == "15 minutes" and run["props"]["Trade direction"] == "Short only") or (run["pair"] == "GBPJPY" and run["props"]["Timeframe"] == "5 minutes" and run["props"]["Entry model"] == "Break + retest")
    if selected:
        for subset,summary in run["timing_diagnostic"].items():
            timing.append({"pair":run["pair"],"subset":subset,"closed_trades":summary["n"],"budget_PF":summary["pf"]})
pd.DataFrame(timing).round(3)

,pair,subset,closed_trades,budget_PF
0,GBPJPY,inside_before90,11,5.932
1,GBPJPY,inside_after90,13,0.504
2,GBPJPY,outside_session,0,NaN
3,XAUUSD,inside_before90,8,0.625
4,XAUUSD,inside_after90,25,2.283
5,XAUUSD,outside_session,0,NaN


### 5. Paper-trade uncertainty

Consolidated closed trades, not partial-close events. No original planned risk/setup labels are assumed.

In [5]:
s=paper["closed_trades"]
pd.DataFrame([{"closed_trades":s["n"],"wins":s["wins"],"net_ZAR":s["net"],"PF":s["pf"],"win_pct":s["win_pct"],"Wilson_95_low_pct":s["ci_low"],"Wilson_95_high_pct":s["ci_high"],"net_without_best_ZAR":s["net_without_best"]}]).round(3)

,closed_trades,wins,net_ZAR,PF,win_pct,Wilson_95_low_pct,Wilson_95_high_pct,net_without_best_ZAR
0,10,5,4890.11,3.098,50.0,23.659,76.341,1563.5


## Takeaways

- Prioritize EURUSD M5 London, GBPJPY M5 London, GBPUSD M5 London retest, and XAUUSD M15 NY short as hypotheses. Test both-direction controls.
- Do not universally impose a 90-minute delay: the XAUUSD and GBPJPY subsets point in different directions.
- Keep feed/dates/costs/stop/target/risk constant, freeze settings before an untouched period, and record all forward signals, including skips.
- A week is a usability review, not statistical validation. See `README.md` and `../ORB_V3_SETUP_GUIDE.md` for definitions, caveats, implementation and upload instructions.

Platform sources: [Strategy Tester execution](https://www.tradingview.com/pine-script-docs/concepts/strategies/), [timezones](https://www.tradingview.com/pine-script-docs/concepts/time/).